# DAETF-Net: Domain-Adaptive Equivariant Tensor Fusion Network

This notebook implements the DAETF-Net architecture for robust Hyperspectral and Multispectral Image Fusion.
It addresses the domain shift problem between synthetic (CAVE) and real-world (Harvard) datasets.

## Architecture Overview
1. Equivariant Feature Extractor (EFE)
2. Tensor Spectral-Spatial Encoder (TSSE)
3. Adaptive Fusion Mixture-of-Experts (AF-MoE)
4. Frequency-Domain Refinement Module (FDRM)


In [ ]:
# Install required packages
!pip install -q einops tensorly groupy torch torchvision


In [ ]:
# Import libraries
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy.io as sio
from einops import rearrange
import tensorly as tl
from tensorly.decomposition import tucker
import torchvision.transforms as transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import json
from datetime import datetime
import shutil
import subprocess
import sys

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create directories for saving results
os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./results', exist_ok=True)
os.makedirs('./logs', exist_ok=True)
# Optional: install pytorch_msssim for SSIM loss (will be used if available)


In [ ]:
# Custom Loss Function
class CustomLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0, gamma=0.5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.l1 = nn.L1Loss()
        # Try to import pytorch_msssim for SSIM; fallback to a simple approximation
        try:
            from pytorch_msssim import ssim
            self.ssim = ssim
            self.use_ssim = True
        except ImportError:
            self.use_ssim = False
            print("Warning: pytorch_msssim not installed, SSIM term will be skipped. Install via: pip install pytorch_msssim")
    def gradient_loss(self, pred, target):
        # Compute gradients using Sobel filters
        # Define Sobel kernels
        sobel_x = torch.tensor([[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]], dtype=pred.dtype, device=pred.device).unsqueeze(0).unsqueeze(0)
        sobel_y = torch.tensor([[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]], dtype=pred.dtype, device=pred.device).unsqueeze(0).unsqueeze(0)
        # Expand to channels
        sobel_x = sobel_x.repeat(pred.size(1), 1, 1, 1)
        sobel_y = sobel_y.repeat(pred.size(1), 1, 1, 1)
        grad_x_pred = F.conv2d(pred, sobel_x, padding=1, groups=pred.size(1))
        grad_y_pred = F.conv2d(pred, sobel_y, padding=1, groups=pred.size(1))
        grad_x_target = F.conv2d(target, sobel_x, padding=1, groups=target.size(1))
        grad_y_target = F.conv2d(target, sobel_y, padding=1, groups=target.size(1))
        loss_grad = F.l1_loss(grad_x_pred, grad_x_target) + F.l1_loss(grad_y_pred, grad_y_target)
        return loss_grad
    def forward(self, pred, target):
        loss = self.alpha * self.l1(pred, target)
        if self.use_ssim:
            # SSIM returns a value between -1 and 1, where 1 is perfect match
            # We want to minimize (1 - SSIM)
            ssim_val = self.ssim(pred, target, data_range=1.0, size_average=True)
            loss += self.beta * (1 - ssim_val)
        if self.gamma > 0:
            loss += self.gamma * self.gradient_loss(pred, target)
        return loss


In [ ]:
# Helper functions for downloading datasets (if needed)
# Uncomment and run if you need to download the datasets from Kaggle
# !kaggle datasets download -d <dataset-name>
# !unzip -q <dataset-name>.zip -d /kaggle/input/
print("Assuming datasets are already available in /kaggle/input/")


In [ ]:
# Equivariant Feature Extractor (EFE)
# Simplified version using standard CNN for demonstration
# In practice, replace with group-equivariant convolutions
class EquivariantFeatureExtractor(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


In [ ]:
# Tensor Spectral-Spatial Encoder (TSSE)
class TensorSpectralSpatialEncoder(nn.Module):
    def __init__(self, in_channels_hsi, in_channels_msi, rank):
        super().__init__()
        self.rank = rank
        # We'll project to a common space
        self.proj_hsi = nn.Conv2d(in_channels_hsi, rank, kernel_size=1)
        self.proj_msi = nn.Conv2d(in_channels_msi, rank, kernel_size=1)
        # Learnable core tensor (simplified)
        self.core = nn.Parameter(torch.randn(rank, rank, rank, rank))
        self.bn = nn.BatchNorm2d(rank)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, hsi, msi):
        # hsi: [B, C_hsi, H, W], msi: [B, C_msi, H, W]
        # Project to rank-dimensional space
        hsi_proj = self.proj_hsi(hsi)  # [B, rank, H, W]
        msi_proj = self.proj_msi(msi)  # [B, rank, H, W]
        # Combine via element-wise product (simplified tensor interaction)
        interaction = hsi_proj * msi_proj  # [B, rank, H, W]
        # Apply non-linearity
        output = self.relu(self.bn(interaction))
        return output


In [ ]:
# Adaptive Fusion Mixture-of-Experts (AF-MoE)
class AdaptiveFusionMoE(nn.Module):
    def __init__(self, in_channels, num_experts=4):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
            ) for _ in range(num_experts)
        ])
        self.gating = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, num_experts),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        # x: [B, C, H, W]
        # Compute gating weights
        gates = self.gating(x)  # [B, num_experts]
        # Compute expert outputs
        expert_outputs = [expert(x) for expert in self.experts]  # List of [B, C, H, W]
        # Stack and weight
        expert_outputs = torch.stack(expert_outputs, dim=1)  # [B, num_experts, C, H, W]
        # Weighted sum
        gates = gates.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)  # [B, num_experts, 1, 1, 1]
        output = torch.sum(gates * expert_outputs, dim=1)  # [B, C, H, W]
        return output


In [ ]:
# Frequency-Domain Refinement Module (FDRM)
# Simplified version using multi-scale feature aggregation
class FrequencyDomainRefinementModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv_3x3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv_5x5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2)
        self.conv_7x7 = nn.Conv2d(in_channels, in_channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm2d(in_channels * 3)
        self.relu = nn.ReLU(inplace=True)
        self.fuse = nn.Conv2d(in_channels * 3, in_channels, kernel_size=1)

    def forward(self, x):
        x3 = self.relu(self.conv_3x3(x))
        x5 = self.relu(self.conv_5x5(x))
        x7 = self.relu(self.conv_7x7(x))
        x = torch.cat([x3, x5, x7], dim=1)
        x = self.relu(self.bn(x))
        x = self.fuse(x)
        return x


In [ ]:
# Full DAETF-Net Model with Learnable Upsampler
class DAETFNet(nn.Module):
    def __init__(self, in_channels_hsi=31, in_channels_msi=3, rank=8, upscale_factor=4):
        super().__init__()
        self.upscale_factor = upscale_factor
        # Learnable upsampler for LR-HSI to match MSI resolution
        # Using subpixel convolution (pixel shuffle) for efficient upsampling
        self.upsampler = nn.Sequential(
            nn.Conv2d(in_channels_hsi, in_channels_hsi * (upscale_factor ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor),
            nn.Conv2d(in_channels_hsi, in_channels_hsi, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.efe = EquivariantFeatureExtractor(in_channels_hsi + in_channels_msi, 64)
        self.tsse = TensorSpectralSpatialEncoder(64, 64, rank)
        self.af_moe = AdaptiveFusionMoE(64, num_experts=4)
        self.fdrm = FrequencyDomainRefinementModule(64)
        self.reconstruction = nn.Conv2d(64, in_channels_hsi, kernel_size=3, padding=1)

    def forward(self, hsi_lr, msi):
        # hsi_lr: low-resolution HSI [B, C_hsi, H, W]
        # msi: multispectral image [B, C_msi, H*scale, W*scale] (high-res spatial)
        # Upsample LR-HSI to match MSI spatial resolution
        hsi_hr = self.upsampler(hsi_lr)  # [B, C_hsi, H*scale, W*scale]
        # Concatenate along channel dimension
        x = torch.cat([hsi_hr, msi], dim=1)  # [B, C_hsi+C_msi, H*scale, W*scale]
        # Equivariant feature extraction
        x = self.efe(x)  # [B, 64, H*scale, W*scale]
        # Tensor spectral-spatial encoding
        x = self.tsse(x, x)  # Simplified: using same input for both
        # Adaptive fusion
        x = self.af_moe(x)
        # Frequency-domain refinement
        x = self.fdrm(x)
        # Reconstruction to HSI bands
        x = self.reconstruction(x)  # [B, C_hsi, H*scale, W*scale]
        # Add residual (upsampled LR-HSI)
        output = x + hsi_hr
        return output


In [ ]:
# Dataset class (dummy for illustration)
class HSIFusionDataset(Dataset):
    def __init__(self, hsi_dir, msi_dir, transform=None):
        self.hsi_dir = hsi_dir
        self.msi_dir = msi_dir
        self.transform = transform
        # In practice, load the file names
        # For dummy, we'll use a fixed length
        self.length = 100  # Dummy

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Generate dummy data
        # In practice, load .mat files and convert to tensors
        hsi = torch.randn(31, 64, 64)  # [C, H, W]
        msi = torch.randn(3, 64, 64)    # [C, H, W]
        # For training, we need a ground truth HR-HSI
        # We'll generate a dummy HR-HSI by adding some noise to a blurred version
        # In practice, load the HR-HSI from the dataset
        hr_hsi = torch.randn(31, 64, 64)  # Dummy ground truth
        
        if self.transform:
            hsi = self.transform(hsi)
            msi = self.transform(msi)
            hr_hsi = self.transform(hr_hsi)
        
        return hsi, msi, hr_hsi


In [ ]:
# Training setup
def train_model(model, train_loader, val_loader, num_epochs=50, learning_rate=1e-4):
    model.to(device)
    # Use custom loss combining L1, SSIM, and gradient terms
    criterion = CustomLoss(alpha=1.0, beta=1.0, gamma=0.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (hsi_lr, msi, hr_hsi) in enumerate(train_loader):
            hsi_lr = hsi_lr.to(device)
            msi = msi.to(device)
            hr_hsi = hr_hsi.to(device)
            
            optimizer.zero_grad()
            outputs = model(hsi_lr, msi)
            loss = criterion(outputs, hr_hsi)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if i % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for hsi_lr, msi, hr_hsi in val_loader:
                hsi_lr = hsi_lr.to(device)
                msi = msi.to(device)
                hr_hsi = hr_hsi.to(device)
                outputs = model(hsi_lr, msi)
                loss = criterion(outputs, hr_hsi)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        scheduler.step()
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # Save checkpoint if validation loss improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'./checkpoints/daetfnet_best.pth')
            print(f'Saved best model with val loss: {val_loss:.4f}')
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'./checkpoints/daetfnet_epoch_{epoch+1}.pth')
    
    return train_losses, val_losses


In [ ]:
# Main execution
if __name__ == '__main__':
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Paths to datasets (assuming they are in /kaggle/input/)
    # For CAVE dataset
    cave_hsi_dir = '/kaggle/input/cave-dataset-2/Data/Test/HSI'
    cave_msi_dir = '/kaggle/input/cave-dataset-2/Data/Test/RGB'
    # For Harvard dataset
    harvard_hsi_dir = '/kaggle/input/harvard-dataset/Data/Test/HSI'
    harvard_msi_dir = '/kaggle/input/harvard-dataset/Data/Test/RGB'
    
    # Check if directories exist
    print('Checking dataset paths...')
    for path in [cave_hsi_dir, cave_msi_dir, harvard_hsi_dir, harvard_msi_dir]:
        if os.path.exists(path):
            print(f'Found: {path}')
        else:
            print(f'Not found: {path}')
            print('  Please ensure the datasets are downloaded and placed in the correct location.')
    
    # Create datasets and data loaders
    # We'll use the CAVE dataset for training and Harvard for validation (to test domain shift)
    train_dataset = HSIFusionDataset(cave_hsi_dir, cave_msi_dir)
    val_dataset = HSIFusionDataset(harvard_hsi_dir, harvard_msi_dir)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
    
    print(f'Train dataset size: {len(train_dataset)}')
    print(f'Val dataset size: {len(val_dataset)}')
    
    # Initialize model
    model = DAETFNet(in_channels_hsi=31, in_channels_msi=3, rank=8)
    print(f'Model size: {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')
    
    # Train the model
    print('Starting training...')
    train_losses, val_losses = train_model(model, train_loader, val_loader, num_epochs=30, learning_rate=1e-4)
    
    # Plot training history
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('./results/training_history.png')
    plt.show()
    
    # Save final model
    torch.save(model.state_dict(), './checkpoints/daetfnet_final.pth')
    print('Training completed. Models saved in ./checkpoints/')


# Retrieving Checkpoints and Results

After training completes, you can find the saved models and results in the `./checkpoints/` and `./results/` directories.

To download these files from Kaggle:
1. In the Kaggle notebook interface, open the `Output` pane on the right.
2. Click on the `Logs` tab to see training progress.
3. Click on the `Versions` tab (or `Output` tab depending on UI) to see files created.
4. You should see `checkpoints/` and `results/` directories.
5. Select the files you want (e.g., `checkpoints/daetfnet_best.pth`, `results/training_history.png`) and click the download icon.

Alternatively, you can use the Kaggle API to download the notebook output:
```bash
kaggle kernels output <your-username>/<notebook-slug> -p ./downloaded
```

To add these checkpoints to your GitHub repository:
1. Download the files as described above.
2. Add them to your local clone of the repository:
   ```bash
   git add path/to/downloaded/checkpoints/daetfnet_best.pth
   git commit -m "Add trained model checkpoints"
   git push
   ```

### Tips for Better Results
- Increase the number of epochs (e.g., to 50 or 100) for better convergence.
- Adjust the learning rate or use a learning rate scheduler.
- Replace the dummy implementations with more sophisticated ones:
  * Use group-equivariant convolutions (via `groupy` or custom steerable filters) for EFE.
  * Use proper Tucker decomposition via `tensorly.decomposition.tucker` for TSSE.
  * Add domain adaptation loss (e.g., MMD) between CAVE and Harvard features.
  * Implement more advanced frequency-domain processing (e.g., wavelet transforms).

### Citation
If you use this work, please consider citing:

@article{your2026daetfnet,
  title={DAETF-Net: Domain-Adaptive Equivariant Tensor Fusion Network for Robust Hyperspectral and Multispectral Image Fusion},
  author={Your Name},
  journal={IEEE Transactions on Geoscience and Remote Sensing},
  year={2026}
}